# **Regression**

## Objectives

*   Fit and evaluate a regression model to predict SalePrice


## Inputs

* outputs/datasets/collection/house_prices_records.csv
* Instructions on which variables to use for data cleaning and feature engineering. They are found in their respective notebooks.

## Outputs

* Train set (features and target)
* Test set (features and target)
* ML pipeline to predict SalePrice
* labels map
* Feature Importance Plot



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [ ]:
import os
current_dir = os.getcwd()
current_dir

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

Confirm the new current directory

In [ ]:
current_dir = os.getcwd()
current_dir

# Load Data

In [ ]:
import numpy as np
import pandas as pd
df = (pd.read_csv("outputs/datasets/collection/house_prices_records.csv")
      )

print(df.shape)
df.head(3)


In [ ]:
df_modeling = df.copy()
df_modeling.head(3)

We add the binary flags (HasOpenPorch, HasMasonry) before the pipeline.
Pipelines do not support dynamic column creation inside by default — so we do this outside and feed into TrainSet.

In [ ]:
# Create binary indicator columns
df_modeling['HasOpenPorch'] = (df_modeling['OpenPorchSF'] > 0).astype(int)
df_modeling['HasMasonry'] = (df_modeling['MasVnrArea'] > 0).astype(int)


---

# MP Pipeline: Regressor

## Create ML pipeline

Note: 

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectFromModel
from feature_engine.imputation import MeanMedianImputer, CategoricalImputer, ArbitraryNumberImputer
from feature_engine.selection import DropFeatures
from feature_engine.encoding import OrdinalEncoder
from feature_engine.transformation import (
    PowerTransformer, BoxCoxTransformer, YeoJohnsonTransformer, LogTransformer
)
# ML algorithms
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import AdaBoostRegressor
from sklearn.ensemble import ExtraTreesRegressor


def PipelineOptimization(model):
    # MODELING PIPELINE
    modeling_pipeline = Pipeline([
    
        # --- 1. Data Cleaning ---
        ("drop_features", DropFeatures(features_to_drop=["EnclosedPorch", "WoodDeckSF"])),
        ("median_imputer", MeanMedianImputer(imputation_method="median", variables=[
            "2ndFlrSF", "BedroomAbvGr", "LotFrontage"
        ])),
        ("constant_imputer", ArbitraryNumberImputer(arbitrary_number=1900.0, variables=["GarageYrBlt"])),
        ("mode_imputer", CategoricalImputer(imputation_method="frequent", variables=[
            "BsmtExposure", "BsmtFinType1", "GarageFinish"
        ])),

        # --- 2. Feature Engineering ---
        # 2.1 Drop highly sparse features (already transformed into binary outside)
        ("drop_sparse", DropFeatures(features_to_drop=["OpenPorchSF", "MasVnrArea"])),

        # 2.2 Ordinal Encoding
        ("ordinal_encoder", OrdinalEncoder(encoding_method="arbitrary", variables=[
            "BsmtExposure", "BsmtFinType1", "GarageFinish", "KitchenQual"
        ])),

        # 2.3 Numerical Transformation
        ("power_1stFlrSF", PowerTransformer(variables=["1stFlrSF"])),
        ("yeojohnson_group1", YeoJohnsonTransformer(variables=[
            "BedroomAbvGr", "BsmtFinSF1", "BsmtUnfSF", "GarageArea", "YearBuilt", "YearRemodAdd",
            "GrLivArea", "LotArea", "LotFrontage", "OverallCond", "TotalBsmtSF"
        ])),
        ("log_garageyrblt", LogTransformer(variables=["GarageYrBlt"], base="10")),
        ("boxcox_overallqual", BoxCoxTransformer(variables=["OverallQual"])),

        # 2.4 Drop correlated features (manual protection of important features)
        ("drop_correlated", DropFeatures(features_to_drop=["1stFlrSF", "GarageYrBlt"])),

        # --- 3. Feature Scaling ---
        ("feat_scaling", StandardScaler()),

        # --- 4. Feature Selection (e.g., LassoCV, RandomForest etc.) ---
        ("feat_selection", SelectFromModel(estimator=model)),

        # --- 5. Final Model ---
        ("model", model)
    ])

    return modeling_pipeline


Custom Class for hyperparameter optimisation

In [ ]:
from sklearn.model_selection import GridSearchCV


class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")
            model = PipelineOptimization(self.models[key])

            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring)
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)

        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]

        return df[columns], self.grid_searches


---

## Split Train Test Set

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    df.drop(['SalePrice'], axis=1),
    df['SalePrice'],
    test_size=0.2,
    random_state=0
)

print("* Train set:", X_train.shape, y_train.shape,
      "\n* Test set:",  X_test.shape, y_test.shape)


## Grid Search CV - Sklearn

### Use default hyperparameters to find most suitable algorithm

In [ ]:
models_quick_search = {
    'LinearRegression': LinearRegression(),
    "DecisionTreeRegressor": DecisionTreeRegressor(random_state=0),
    "RandomForestRegressor": RandomForestRegressor(random_state=0),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
    "AdaBoostRegressor": AdaBoostRegressor(random_state=0),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=0),
    "XGBRegressor": XGBRegressor(random_state=0),
}

params_quick_search = {
    'LinearRegression': {},
    "DecisionTreeRegressor": {},
    "RandomForestRegressor": {},
    "ExtraTreesRegressor": {},
    "AdaBoostRegressor": {},
    "GradientBoostingRegressor": {},
    "XGBRegressor": {},
}

Do a hyperparameter optimisation search using default hyperparameters

In [ ]:
search = HyperparameterOptimizationSearch(models=models_quick_search, params=params_quick_search)
search.fit(X_train, y_train, scoring='r2', n_jobs=-1, cv=5)

Check results

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

ExtraTreesRegressor has the highest average R² but also the highest variance (std = 0.066). This could indicate that it’s sensitive to data variation.

LinearRegression shows stable performance (low std = 0.0377) and does surprisingly well, meaning the engineered features capture linear trends effectively.

XGBRegressor and DecisionTreeRegressor underperform here, which might suggest that either:

* The hyperparameters are suboptimal,

* The models are overfitting (especially for trees),

* Or the ensemble power of RandomForest/ExtraTrees outperforms single models.

### Do an extensive search on the most suitable model (ExtraTreesRegressor) to find the best hyperparameter configuration.

Define model and parameters, for Extensive Search

In [ ]:
models_search = {
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=0),
}

# documentation to help on hyperparameter list: 
# https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.ExtraTreesRegressor.html

# We will not conduct an extensive search, since the focus
# is on how to combine all knowledge in an applied project.
# In a workplace project, you may consider more hyperparameters and spend more time in this step

params_search = {
    "ExtraTreesRegressor": {
        'model__n_estimators': [100, 150, 200],
        'model__max_depth': [5, 10, 15, 20],
        'model__min_samples_split': [5, 10, 15],
        'model__min_samples_leaf': [2, 4, 6],
        'model__max_features': ['sqrt', 'log2'],
        'model__criterion': ['absolute_error']
    }
}

Extensive GridSearch CV

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train, scoring = 'r2', n_jobs=-1, cv=5, verbose=1)

Check results

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

Check the best model

In [ ]:
best_model_extra_trees = grid_search_summary.iloc[0, 0]
best_model_extra_trees

Parameters for best model

In [ ]:
grid_search_pipelines[best_model_extra_trees].best_params_

Define the best regressor, based on search

In [ ]:
best_regressor_pipeline_extra_trees = grid_search_pipelines[best_model_extra_trees].best_estimator_
best_regressor_pipeline_extra_trees

## Assess feature importance

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style('whitegrid')

# Extract steps before scaling (up to but not including 'feat_selection')
data_cleaning_feat_eng_steps = 12
X_transformed = Pipeline(best_regressor_pipeline_extra_trees.steps[:data_cleaning_feat_eng_steps]).transform(X_train)

# Manually assign column names from X_train, after all previous transformations
# (Assumes that the transformation preserves column order)
columns_before_selection = X_train.columns

# If your pipeline drops/adds columns, it's safer to extract names before scaling:
# Step 1: run pipeline up to just before 'feat_scaling'
pipeline_before_scaling = Pipeline(best_regressor_pipeline_extra_trees.steps[:11])  # up to 'drop_correlated'
X_engineered = pipeline_before_scaling.transform(X_train)

# Step 2: wrap in DataFrame to keep column names
X_engineered_df = pd.DataFrame(X_engineered, columns=X_engineered.columns)

# Step 3: get support from SelectFromModel
mask = best_regressor_pipeline_extra_trees['feat_selection'].get_support()
best_features = X_engineered_df.columns[mask].to_list()

# Step 4: plot feature importances
df_feature_importance = pd.DataFrame({
    'Feature': best_features,
    'Importance': best_regressor_pipeline_extra_trees['model'].feature_importances_
}).sort_values(by='Importance', ascending=False)

print(f"* These are the {len(best_features)} most important features in descending order:\n"
      f"{df_feature_importance['Feature'].to_list()}")

# Plot
df_feature_importance.plot(kind='bar', x='Feature', y='Importance')
plt.show()


### Do an extensive search on Ridge and Lasso to find the best hyperparameter configuration.

Define model and parameters, for Extensive Search

In [ ]:
from sklearn.linear_model import Ridge, Lasso

models_search = {
    "Ridge": Ridge(random_state=0),
    "Lasso": Lasso(random_state=0),
}

params_search = {
    "Ridge": {
        'model__alpha': [0.01, 0.1, 1.0, 10.0, 100.0],
        'model__solver': ['auto', 'svd', 'cholesky', 'sparse_cg']
    },
    "Lasso": {
        'model__alpha': [0.01, 0.1, 1.0, 10.0],
        'model__selection': ['cyclic', 'random'],
        'model__max_iter': [1000, 5000]
    }
}

Extensive GridSearch CV

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train, scoring = 'r2', n_jobs=-1, cv=3, verbose=1)

Check results

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

Check the best model

In [ ]:
best_model_lasso = grid_search_summary.iloc[0, 0]
best_model_lasso

Parameters for best model

In [ ]:
grid_search_pipelines[best_model_lasso].best_params_

Define the best regressor, based on search

In [ ]:
best_regressor_pipeline_lasso = grid_search_pipelines[best_model_lasso].best_estimator_
best_regressor_pipeline_lasso

## Coefficient analysis

In [ ]:
def linear_model_coefficients(model, columns):
    print(f"* Interception: {model.intercept_}")
    
    coeff_df = pd.DataFrame({
        'Feature': columns,
        'Coefficient': model.coef_
    }).sort_values(by='Coefficient', key=abs, ascending=False)

    print("* Coefficients")
    print(coeff_df)

    # Plot
    coeff_df.plot(kind='bar', x='Feature', y='Coefficient')
    plt.title("Feature Importance (Linear Model Coefficients)")
    plt.tight_layout()
    plt.show()


In [ ]:
# Step 1: Run pipeline up to just before 'feat_selection'
pipeline_before_selection = Pipeline(best_regressor_pipeline_lasso.steps[:12])
X_engineered = pipeline_before_selection.transform(X_train)

# Step 2: Infer correct column names dynamically
# This assumes you're using feature_engine transformers which store variables_
feature_names = []

for name, step in pipeline_before_selection.steps:
    if hasattr(step, 'variables_'):
        feature_names = step.variables_
    elif hasattr(step, 'get_feature_names_out'):
        feature_names = step.get_feature_names_out()
    # fall back to current columns
    elif isinstance(X_train, pd.DataFrame):
        feature_names = X_train.columns

# Now verify shape matches
X_engineered_df = pd.DataFrame(X_engineered, columns=feature_names)

# Step 3: Get selected features by name
mask = best_regressor_pipeline_lasso['feat_selection'].get_support()
selected_features = X_engineered_df.columns[mask].to_list()

# Step 4: Plot and print coefficients
linear_model_coefficients(best_regressor_pipeline_lasso['model'], selected_features)

### Do an extensive search on RandomForestRegressor model to find the best hyperparameter configuration.

Define model and parameters, for Extensive Search

In [ ]:
from sklearn.ensemble import RandomForestRegressor

models_search = {
    "RandomForestRegressor": RandomForestRegressor(random_state=0),
}

# Reference: https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html
params_search = {
    "RandomForestRegressor": {
        'model__n_estimators': [100, 200, 300],
        'model__max_depth': [None, 10, 20, 30],
        'model__min_samples_split': [2, 5],
        'model__min_samples_leaf': [1, 2],
        'model__max_features': ['sqrt', 'log2'],
        'model__criterion': ['squared_error', 'absolute_error'],
    }
}


Extensive GridSearch CV

In [ ]:
search = HyperparameterOptimizationSearch(models=models_search, params=params_search)
search.fit(X_train, y_train, scoring = 'r2', n_jobs=-1, cv=3, verbose=1)

Check results

In [ ]:
grid_search_summary, grid_search_pipelines = search.score_summary(sort_by='mean_score')
grid_search_summary

Check the best model

In [ ]:
best_model_random_forest = grid_search_summary.iloc[0, 0]
best_model_random_forest

Parameters for best model

In [ ]:
grid_search_pipelines[best_model_random_forest].best_params_

Define the best regressor, based on search

In [ ]:
best_regressor_pipeline_random_forest = grid_search_pipelines[best_model_random_forest].best_estimator_
best_regressor_pipeline_random_forest

## Assess feature importance

In [ ]:

# Extract steps before scaling (up to but not including 'feat_selection')
data_cleaning_feat_eng_steps = 12
X_transformed = Pipeline(best_regressor_pipeline_random_forest.steps[:data_cleaning_feat_eng_steps]).transform(X_train)

# Manually assign column names from X_train, after all previous transformations
# (Assumes that the transformation preserves column order)
columns_before_selection = X_train.columns

# If your pipeline drops/adds columns, it's safer to extract names before scaling:
# Step 1: run pipeline up to just before 'feat_scaling'
pipeline_before_scaling = Pipeline(best_regressor_pipeline_random_forest.steps[:11])  # up to 'drop_correlated'
X_engineered = pipeline_before_scaling.transform(X_train)

# Step 2: wrap in DataFrame to keep column names
X_engineered_df = pd.DataFrame(X_engineered, columns=X_engineered.columns)

# Step 3: get support from SelectFromModel
mask = best_regressor_pipeline_random_forest['feat_selection'].get_support()
best_features = X_engineered_df.columns[mask].to_list()

# Step 4: plot feature importances
df_feature_importance = pd.DataFrame({
    'Feature': best_features,
    'Importance': best_regressor_pipeline_random_forest['model'].feature_importances_
}).sort_values(by='Importance', ascending=False)

print(f"* These are the {len(best_features)} most important features in descending order:\n"
      f"{df_feature_importance['Feature'].to_list()}")

# Plot
df_feature_importance.plot(kind='bar', x='Feature', y='Importance')
plt.show()


## Model Selection Justification: ExtraTreesRegressor

After evaluating multiple models including Ridge, Lasso, and RandomForestRegressor, the ExtraTreesRegressor was selected as the final estimator for the following key reasons:

* Best Overall Cross-Validation Score
    * ExtraTreesRegressor achieved the highest mean cross-validation R² score: 0.8264, compared to:

        * Lasso: 0.8147

        * Ridge: 0.7762

        * RandomForestRegressor: 0.7853

    * It also maintained a low standard deviation (≈ 0.0345), indicating stable performance across all folds.

* Extra Trees introduces additional randomness during the split selection process (random thresholds instead of best thresholds), which:

    * Reduces variance compared to standard Random Forests.

    * Provides better generalization to unseen data.

* Efficient and Scalable

    * The model is well-suited for datasets with many features and handles both numerical and categorical data (after encoding) efficiently.

    * It's parallelizable and scales better for grid search compared to some other ensemble models.

* Outperformed Alternatives in Grid Search

    Compared to:

    *  Lasso, which performed well but lacked the ability to capture non-linear relationships.

    * Ridge, which consistently underperformed.

    * RandomForestRegressor, which had higher variance (std ≈ 0.056–0.092), and lower mean scores.

**Conclusion**

ExtraTreesRegressor was chosen for its strong predictive performance, low variance, and robustness to overfitting. It offered the best trade-off between bias and variance, making it an ideal choice for the regression task of predicting Ames house prices.



## Evaluate on Train and Test Sets

In [ ]:
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
import numpy as np


def regression_performance(X_train, y_train, X_test, y_test, pipeline):
    print("Model Evaluation \n")
    print("* Train Set")
    regression_evaluation(X_train, y_train, pipeline)
    print("* Test Set")
    regression_evaluation(X_test, y_test, pipeline)


def regression_evaluation(X, y, pipeline):
    prediction = pipeline.predict(X)
    print('R2 Score:', r2_score(y, prediction).round(3))
    print('Mean Absolute Error:', mean_absolute_error(y, prediction).round(3))
    print('Mean Squared Error:', mean_squared_error(y, prediction).round(3))
    print('Root Mean Squared Error:', np.sqrt(
        mean_squared_error(y, prediction)).round(3))
    print("\n")


def regression_evaluation_plots(X_train, y_train, X_test, y_test, pipeline, alpha_scatter=0.5):
    pred_train = pipeline.predict(X_train)
    pred_test = pipeline.predict(X_test)

    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))
    sns.scatterplot(x=y_train, y=pred_train, alpha=alpha_scatter, ax=axes[0])
    sns.lineplot(x=y_train, y=y_train, color='red', ax=axes[0])
    axes[0].set_xlabel("Actual")
    axes[0].set_ylabel("Predictions")
    axes[0].set_title("Train Set")

    sns.scatterplot(x=y_test, y=pred_test, alpha=alpha_scatter, ax=axes[1])
    sns.lineplot(x=y_test, y=y_test, color='red', ax=axes[1])
    axes[1].set_xlabel("Actual")
    axes[1].set_ylabel("Predictions")
    axes[1].set_title("Test Set")

    plt.show()


Evaluate Performance

In [ ]:
regression_performance(X_train, y_train, X_test, y_test, best_regressor_pipeline_extra_trees)
regression_evaluation_plots(X_train, y_train, X_test, y_test, best_regressor_pipeline_extra_trees)

**Overfitting Mitigation Strategy**

During initial evaluation, the ExtraTreesRegressor demonstrated strong performance on the training set but exhibited signs of overfitting, as seen from the gap in R² scores between the training and test sets.

To reduce this, the following steps were taken:

* Regularization via hyperparameter tuning

The initial model used relatively loose parameters:
```
'model__max_depth': [None, 10, 20, 30],
'model__min_samples_split': [2, 5, 10],
'model__min_samples_leaf': [1, 2, 4],
'model__criterion': ['squared_error', 'absolute_error']
```
These were adjusted to enforce regularization:
```
'model__max_depth': [5, 10, 15, 20],
'model__min_samples_split': [5, 10, 15],
'model__min_samples_leaf': [2, 4, 6],
'model__criterion': ['absolute_error'],  # Dropped squared_error, to reduce sensitivity to outliers.
```
* Reduced model complexity
    * Removed very deep trees (None) and limited growth to reduce variance.
    * Increased min_samples_leaf and min_samples_split to require more data per split and leaf.

* Improved cross-validation stability

    Cross-validation folds increased from 3 to 5, offering a more reliable estimate of generalization and reducing the chance of optimistic bias.

**Result**

After applying the changes:

* The model's training R² dropped slightly (from 0.96 to 0.90), indicating reduced overfitting.

* The test R² remained strong at 0.812, confirming improved generalization.

* The gap between train and test performance narrowed, with MAE and RMSE remaining within acceptable ranges.

This final version strikes a balance between bias and variance, and is well-suited for real-world prediction tasks on unseen data.


---

# Push files to Repo

We will generate the following files

* Train set
* Test set
* Modeling pipeline
* features importance plot

In [ ]:
import joblib
import os

version = 'v1'
file_path = f'outputs/ml_pipeline/predict_sale_price/{version}'

try:
  os.makedirs(name=file_path)
except Exception as e:
  print(e)

## Train Set: features and target

In [ ]:
X_train.head()

In [ ]:
X_train.to_csv(f"{file_path}/X_train.csv", index=False)

In [ ]:
y_train

In [ ]:
y_train.to_csv(f"{file_path}/y_train.csv", index=False)

## Test Set: features and target

In [ ]:
X_test.head()

In [ ]:
X_test.to_csv(f"{file_path}/X_test.csv", index=False)

In [ ]:
y_test

In [ ]:
y_test.to_csv(f"{file_path}/y_test.csv", index=False)

## Modelling pipeline

ML pipeline for predicting SalePrice

In [ ]:
best_regressor_pipeline_extra_trees

In [ ]:
joblib.dump(value=best_regressor_pipeline_extra_trees, filename=f"{file_path}/extra_trees_regressor_pipeline.pkl")

## Feature importance plot

In [ ]:
df_feature_importance.plot(kind='bar', x='Feature', y='Importance')
plt.show()

In [ ]:
df_feature_importance.plot(kind='bar',x='Feature',y='Importance')
plt.savefig(f'{file_path}/features_importance.png', bbox_inches='tight')